In [1]:
import polars as pl
import lightgbm as lgb

from datetime import datetime

from src.util.constants import PATH_RAW_TRAIN_SET, PATH_RAW_VALIDATE_SET, FIXED_LGB_PARAMETERS, DATA_PATH
from src.util.common import mean_grouped_spearman_correlation
from util.common import load_from_pickle

In [2]:
feature_names = load_from_pickle(DATA_PATH / "raw/feature_names.pkl")
eras = load_from_pickle(DATA_PATH / "raw/eras.pkl")

In [3]:
df_validate = ((pl.read_parquet(PATH_RAW_VALIDATE_SET)
               .filter(pl.col("target").is_not_null()))
               .with_columns(pl.col("era").cast(pl.Int16)))
df_validate = df_validate.drop([col for col in df_validate.columns if "target_" in col] + ["data_type", 'id'])

X_validate = df_validate[feature_names].to_numpy()
y_validate = df_validate['target'].to_numpy()
eras_validate = df_validate['era'].to_numpy()

del df_validate

# Sampling eras and sampling within eras

In [4]:
df_result = pl.DataFrame()
num_boost_rounds = [50, 200]

for sampling_type in ["even", "odd", "within_even", "within_odd", "full"]:
    print(f"{datetime.now().strftime('%H:%M:%S')} . . . Using sampling strategy: {sampling_type} eras...")

    df_train = ((pl.read_parquet(PATH_RAW_TRAIN_SET)
                .filter(pl.col("target").is_not_null()))
                .with_columns(pl.col("era").cast(pl.Int16)))
    df_train = df_train.drop([col for col in df_train.columns if "target_" in col] + ["data_type", 'id'])

    len_before = len(df_train)
    if sampling_type in ["even", "odd"]:
        rest = 0 if sampling_type == "even" else 1
        df_train = df_train.filter(pl.col("era") % 2 == rest)
    elif sampling_type in ["within_even", "within_odd"]:
        offset = 1 if sampling_type == "within_even" else 0
        df_train = df_train.gather_every(2, offset=offset)
    else:
        pass  # leave df_train unchanged
    len_after = len(df_train)
    print(f"Using {len_after} out of {len_before} rows.")


    X_train = df_train[feature_names].to_numpy()
    y_train = df_train["target"].to_numpy()
    del df_train

    for num_leaves in [2**6 - 1, 2**9 - 1]:
        print(f"{datetime.now().strftime('%H:%M:%S')} . . . Training with {num_leaves} leaves...")
        parameters = {
            **FIXED_LGB_PARAMETERS,
            "objective": "regression",
            "num_leaves": num_leaves,
            "num_boost_round": max(num_boost_rounds)
        }

        lgb_train = lgb.Dataset(X_train, label=y_train)

        model = lgb.train(
            params=parameters,
            train_set=lgb_train,
            num_boost_round=parameters["num_boost_round"]
        )

        for num_boost_round in num_boost_rounds:
            corr = mean_grouped_spearman_correlation(
                model.predict(X_validate, num_iteration=num_boost_round),
                y_validate,
                eras_validate
            )

            df_result = df_result.vstack(pl.DataFrame({
                "selection_type": sampling_type,
                "num_leaves": num_leaves,
                "num_boost_round": num_boost_round,
                "corr": corr,
            }))

            print(f"{datetime.now().strftime('%H:%M:%S')} . . . Sampling: {sampling_type}, num_leaves={num_leaves}, num_boost_round={num_boost_round}: correlation = {corr:.5f}.")

12:33:01 . . . Using sampling strategy: even eras...
Using 2746268 out of 1374052 rows.
12:33:06 . . . Training with 63 leaves...
12:41:18 . . . Sampling: even, num_leaves=63, num_boost_round=50: correlation = 0.02610.
12:43:01 . . . Sampling: even, num_leaves=63, num_boost_round=200: correlation = 0.02630.
12:43:01 . . . Training with 511 leaves...
13:01:02 . . . Sampling: even, num_leaves=511, num_boost_round=50: correlation = 0.01958.
13:02:51 . . . Sampling: even, num_leaves=511, num_boost_round=200: correlation = 0.02146.
13:02:51 . . . Using sampling strategy: odd eras...
Using 2746268 out of 1372216 rows.
13:02:54 . . . Training with 63 leaves...
13:10:40 . . . Sampling: odd, num_leaves=63, num_boost_round=50: correlation = 0.02442.
13:12:18 . . . Sampling: odd, num_leaves=63, num_boost_round=200: correlation = 0.02436.
13:12:18 . . . Training with 511 leaves...
13:22:08 . . . Sampling: odd, num_leaves=511, num_boost_round=50: correlation = 0.01981.
13:23:47 . . . Sampling: odd,

In [9]:
print(df_result.select("selection_type", "corr").group_by("selection_type").mean().sort("corr", descending=True))

shape: (5, 2)
┌────────────────┬──────────┐
│ selection_type ┆ corr     │
│ ---            ┆ ---      │
│ str            ┆ f64      │
╞════════════════╪══════════╡
│ even           ┆ 0.023359 │
│ full           ┆ 0.023099 │
│ odd            ┆ 0.022414 │
│ within_odd     ┆ 0.020669 │
│ within_even    ┆ 0.020089 │
└────────────────┴──────────┘


Era sampling looks very promising. Within era sampling seems to lose some of the performance.

Let's use 50% era sampling.